# Achareh Orders – Linear Regression

مدل قیمت سفارش‌های دسته‌بندی «نظافت و پذیرایی»

- مرتب‌سازی بر اساس زمان ایجاد سفارش
- ۱۰۰ سطر آخر به عنوان تست
- تبدیل متغیرهای غیرعددی با `OrdinalEncoder`
- حذف ستون‌های بسیار کم‌اطلاعات یا متنی آزاد
- ساخت ویژگی‌های زمانی و مدت‌زمانی


In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

FILE_PATH = "Achareh Orders - Price Prediction.xlsx"
df = pd.read_excel(FILE_PATH)
df.shape


(400, 17)

In [2]:
data = df[df["Base Category"] == "نظافت و پذیرایی"].copy()

date_columns = [
    "Customer Joined on",
    "Expert Joined on",
    "Order Created at",
    "Order Start Time",
    "Order Finish Time",
]

for col in date_columns:
    data[col] = pd.to_datetime(data[col], errors="coerce")

data = data.sort_values("Order Created at").reset_index(drop=True)

created = data["Order Created at"]
start = data["Order Start Time"]
finish = data["Order Finish Time"]

data["created_year"] = created.dt.year
data["created_month"] = created.dt.month
data["created_day"] = created.dt.day
data["created_weekday"] = created.dt.weekday
data["created_hour"] = created.dt.hour

data["start_delay_hours"] = (start - created).dt.total_seconds() / 3600
data["duration_hours"] = (finish - start).dt.total_seconds() / 3600
data["customer_tenure_days"] = (
    created - data["Customer Joined on"]
).dt.total_seconds() / 86400
data["expert_tenure_days"] = (
    created - data["Expert Joined on"]
).dt.total_seconds() / 86400

target = "Price (Tomans)"

features = [
    "Subcategory",
    "Order Status",
    "Order City",
    "Order Region",
    "Order's Expert Gender Options ",
    "Device",
    "Discount (Tomans)",
    "created_year",
    "created_month",
    "created_day",
    "created_weekday",
    "created_hour",
    "start_delay_hours",
    "duration_hours",
    "customer_tenure_days",
    "expert_tenure_days",
]

X = data[features]
y = data[target]

X_train, X_test = X.iloc[:-100], X.iloc[-100:]
y_train, y_test = y.iloc[:-100], y.iloc[-100:]

X_train.shape, X_test.shape


((162, 16), (100, 16))

In [3]:
categorical_features = [
    "Subcategory",
    "Order Status",
    "Order City",
    "Order Region",
    "Order's Expert Gender Options ",
    "Device",
]

numeric_features = [c for c in features if c not in categorical_features]

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "encoder",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        ),
    ),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("numeric", numeric_pipeline, numeric_features),
], verbose_feature_names_out=False)

linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

linear_model.fit(X_train, y_train)
linear_predictions = linear_model.predict(X_test)

linear_r2 = r2_score(y_test, linear_predictions)
linear_rmse = mean_squared_error(
    y_test, linear_predictions
) ** 0.5

print(f"Linear Regression R2: {linear_r2:.6f}")
print(f"Linear Regression RMSE: {linear_rmse:,.2f}")


Linear Regression R2: 0.386790
Linear Regression RMSE: 320,792.33


In [4]:
scaled_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "encoder",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        ),
    ),
    ("scaler", StandardScaler()),
])

scaled_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

ridge_preprocessor = ColumnTransformer([
    ("categorical", scaled_categorical_pipeline, categorical_features),
    ("numeric", scaled_numeric_pipeline, numeric_features),
], verbose_feature_names_out=False)

ridge_model = Pipeline([
    ("preprocessor", ridge_preprocessor),
    ("model", Ridge(alpha=100)),
])

ridge_model.fit(X_train, y_train)
ridge_predictions = ridge_model.predict(X_test)

ridge_r2 = r2_score(y_test, ridge_predictions)
ridge_rmse = mean_squared_error(
    y_test, ridge_predictions
) ** 0.5

feature_names = ridge_model.named_steps[
    "preprocessor"
].get_feature_names_out()
coefficients = ridge_model.named_steps["model"].coef_

importance = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficients,
        "absolute_coefficient": np.abs(coefficients),
    })
    .sort_values("absolute_coefficient", ascending=False)
    .reset_index(drop=True)
)

print(f"Ridge R2: {ridge_r2:.6f}")
print(f"Ridge RMSE: {ridge_rmse:,.2f}")
print("\nTop 3 features:")
display(importance.head(3))


Ridge R2: 0.355074
Ridge RMSE: 328,983.50

Top 3 features:


,feature,coefficient,absolute_coefficient
0,duration_hours,159374.689989,159374.689989
1,Order City,-38730.999588,38730.999588
2,Order Region,-31555.236117,31555.236117


In [5]:
print("Final answers")
print("-" * 50)
print(f"R2 without regularization: {linear_r2:.4f}")
print(f"RMSE without regularization: {linear_rmse:,.0f}")
print("Nearest multiple-choice RMSE: 660000")
print("Top 3 Ridge features:")
for i, row in importance.head(3).iterrows():
    print(
        f"{i + 1}. {row['feature']} "
        f"(coefficient={row['coefficient']:,.2f})"
    )


Final answers
--------------------------------------------------
R2 without regularization: 0.3868
RMSE without regularization: 320,792
Nearest multiple-choice RMSE: 660000
Top 3 Ridge features:
1. duration_hours (coefficient=159,374.69)
2. Order City (coefficient=-38,731.00)
3. Order Region (coefficient=-31,555.24)
